# Kind-LM Mechanistic Probe v4

## Environment Setup

In [ ]:
import subprocess, sys

packages = [
    "numpy<2.0.0", # pinned: C API binary incompatibility guard (Expected 96, got 88)
    "transformer_lens",
    "einops",
    "plotly",
    "jaxtyping",
    "transformers",
    "pandas",
    "kaleido==0.2.1", # pinned per Plotly compatibility warning
    "scipy", # entropy, FDR correction
    "tuned-lens", # tuned lens cross-validation for Analysis 4
    "sacrebleu", # pairwise lexical diversity for story outputs
    "accelerate", # model parallelism utilities
    ]

for pkg in packages:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=True)

    print(" All packages installed.")

## Imports & Global Config

In [ ]:
import os
import gc
import json
import re
import shutil
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import plotly.express as px
import plotly.graph_objects as go
from scipy.stats import entropy as scipy_entropy
from transformers import GPT2LMHeadModel, AutoTokenizer, GPT2Config
from transformer_lens import HookedTransformer

# Directories
os.makedirs("results_v4/figures", exist_ok=True)
os.makedirs("results_v4/data", exist_ok=True)

# Device
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f" Running on: {DEVICE}")
if DEVICE == "cuda":
    print(f" GPU: {torch.cuda.get_device_name(0)}")
    print(f" VRAM available: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

    # Checkpoint registry (verified against live HuggingFace)
    REPO_BASE = "llm-slice/blm-gpt2s-90M-s42"

    # Dense coverage around the hypothesised threshold (20M-50M words)
    PRETRAINING_REVISIONS = [
        "chck_1M",
        "chck_5M",
        "chck_10M",
        "chck_20M",
        "chck_30M",
        "chck_40M",
        "chck_50M",
        "chck_60M",
        "chck_100M",
        "chck_200M",
        "chck_1000M",
        ]

    # PPO interaction checkpoints - separate repos, canonical seed=42
    PPO_REPOS = {
        "50M+PPO": "llm-slice/blm-gpt2s-90M-s42_chck_50M_ppo-1000K-seed42",
        "200M+PPO": "llm-slice/blm-gpt2s-90M-s42_chck_200M_ppo-1000K-seed42",
        }

    # Checkpoints to run in the logit-lens and coreference analyses
    # (subset to save GPU time - full sweep would take too long)
    MECH_CHECKPOINTS = ["chck_10M", "chck_20M", "chck_50M", "chck_200M", "chck_1000M"]

    # Expected custom vocabulary size - asserted on every load
    EXPECTED_VOCAB = 16_000

    # Story prompt (domain-matched to BabyLM training data)
    STORY_PROMPT = "Once upon a time in a faraway land, there lived a"

    # Parameters
    N_INDUCTION_SAMPLES = 25 # number of random repeated sequences for induction score
    INDUCTION_SEQ_LEN = 30 # half-length of repeated sequence; full sequence = 60 tokens
    EPS_THRESHOLD = 0.5 # KL threshold for "model has committed"

    print(" Config loaded.")

## Robust Model Loading Utility

In [ ]:
def load_model(repo_id: str, revision: str = "main", random_init: bool = False) -> HookedTransformer:
    print(f" Loading {'RANDOM INIT' if random_init else revision} from {repo_id}")

    config_revision = revision if not random_init else "chck_1M"
    tokenizer = AutoTokenizer.from_pretrained(repo_id, revision=config_revision)

    if random_init:
        config = GPT2Config.from_pretrained(repo_id, revision=config_revision)
        hf_model = GPT2LMHeadModel(config)
    else:
        hf_model = GPT2LMHeadModel.from_pretrained(repo_id, revision=revision)

        tl_model = HookedTransformer.from_pretrained(
            "gpt2",
            hf_model=hf_model,
            tokenizer=tokenizer,
            device=DEVICE,
            n_ctx=hf_model.config.n_positions,
            d_vocab=hf_model.config.vocab_size,
            )

        tl_model.hf_model = hf_model.to(DEVICE)

        assert tl_model.cfg.d_vocab == EXPECTED_VOCAB, (
            f" Vocab mismatch! Got {tl_model.cfg.d_vocab}, expected {EXPECTED_VOCAB}. "
            f"Check revision '{revision}' - tokenizer must come from a chck_* branch."
            )

        tl_model.eval()
        return tl_model

        def release(model: HookedTransformer):
            model.reset_hooks(clear_contexts=True)
            del model
            gc.collect()
            if DEVICE == "cuda":
                torch.cuda.empty_cache()
                torch.cuda.synchronize()

                def vram_gb() -> float:
                    if DEVICE == "cuda":
                        return torch.cuda.memory_allocated() / 1e9
                        return 0.0

                        def validate_single_token(model: HookedTransformer, word: str) -> int:
                            tokens = model.to_tokens(word, prepend_bos=False)[0]
                            if len(tokens) > 1:
                                readable = model.to_str_tokens(tokens)
                                raise ValueError(f"'{word}' {len(tokens)} tokens: {readable}")
                                return tokens[0].item()

                                print(" Model loading utilities defined.")

## Analysis 1 - Induction Head Mapping

In [ ]:
def get_induction_scores_robust(
    model: HookedTransformer,
    n_samples: int = N_INDUCTION_SAMPLES,
    seq_len: int = INDUCTION_SEQ_LEN,
) -> torch.Tensor:
    model.reset_hooks()
    n_layers = model.cfg.n_layers
    n_heads = model.cfg.n_heads
    accumulated = torch.zeros(n_layers, n_heads)

    for _ in range(n_samples):
        # Draw random tokens from a safe range well within vocab=16000
        base = torch.randint(100, 15_000, (1, seq_len))
        tokens = torch.cat([base, base], dim=1).to(DEVICE) # [1, 2*seq_len]

        _, cache = model.run_with_cache(
            tokens,
            names_filter=lambda name: "pattern" in name,
            )

        for layer in range(n_layers):
            pattern = cache["pattern", layer][0] # [n_heads, 2*seq_len, 2*seq_len]
            # Average induction attention over all positions in the repeated half
            # Position seq_len+i should attend to position i+1 (the token following
            # the first occurrence of the current token)
            for i in range(seq_len - 1):
                accumulated[layer] += pattern[:, seq_len + i, i + 1].cpu()

                scores = accumulated / (n_samples * (seq_len - 1))
                model.reset_hooks()
                return scores # [n_layers, n_heads] on CPU

# Run across all checkpoints
print(f" Analysis 1: Induction Head Sweep - {N_INDUCTION_SAMPLES} sequences each")
induction_rows = []

# Architectural baseline (random init, 0 words of training)
print(" [0/12] Random init baseline …")
m = load_model(REPO_BASE, random_init=True)
scores = get_induction_scores_robust(m)
induction_rows.append({
    "checkpoint": "0M_random",
    "max_score": scores.max().item(),
    "mean_score": scores.mean().item(),
    "top_head_layer": scores.argmax().item() // m.cfg.n_heads,
    "top_head_idx": scores.argmax().item() % m.cfg.n_heads,
    "scores_json": scores.tolist(),
})
release(m)
print(f" VRAM after release: {vram_gb():.2f} GB")

for i, rev in enumerate(PRETRAINING_REVISIONS, 1):
    print(f" [{i}/{len(PRETRAINING_REVISIONS)}] {rev} …")
    m = load_model(REPO_BASE, revision=rev)
    scores = get_induction_scores_robust(m)
    induction_rows.append({
        "checkpoint": rev.replace("chck_", ""),
        "max_score": scores.max().item(),
        "mean_score": scores.mean().item(),
        "top_head_layer": int(scores.argmax().item() // m.cfg.n_heads),
        "top_head_idx": int(scores.argmax().item() % m.cfg.n_heads),
        "scores_json": scores.tolist(),
        })
    release(m)
    print(f" VRAM after release: {vram_gb():.2f} GB")

df_induction = pd.DataFrame(induction_rows)
df_induction.to_csv("results_v4/data/analysis_1_induction_scores.csv", index=False)
print(" Analysis 1 complete - data saved.")

# Visualise
from plotly.subplots import make_subplots
import plotly.graph_objects as go

fig1 = make_subplots(rows=1, cols=2, subplot_titles=("Peak Induction Score", "Max vs Mean Score"))

fig1.add_trace(go.Scatter(x=df_induction["checkpoint"], y=df_induction["max_score"], mode='lines+markers', name="Max Score"), row=1, col=1)

fig1.add_trace(go.Scatter(x=df_induction["checkpoint"], y=df_induction["max_score"], mode='lines+markers', name="Max Score (dup)"), row=1, col=2)
fig1.add_trace(go.Scatter(x=df_induction["checkpoint"], y=df_induction["mean_score"], mode='lines+markers', name="Mean Score"), row=1, col=2)

fig1.add_vrect(x0="20M", x1="50M", fillcolor="orange", opacity=0.15, annotation_text="Threshold", row=1, col=1)
fig1.add_vrect(x0="20M", x1="50M", fillcolor="orange", opacity=0.15, row=1, col=2)

fig1.update_layout(title_text="Analysis 1: Induction Head Trajectory", template="plotly_white")
fig1.write_html("results_v4/figures/analysis_1_induction.html")
fig1.write_image("results_v4/figures/analysis_1_induction.png", width=1200, height=500)
fig1.show()

## Analysis 2 - PPO Causal Isolation

In [ ]:
print(" Analysis 2: PPO Causal Isolation - Signed Per-Head Delta Heatmaps")

ppo_records = []

for label, (base_rev, ppo_repo) in {
    "50M": ("chck_50M", PPO_REPOS["50M+PPO"]),
    "200M": ("chck_200M", PPO_REPOS["200M+PPO"]),
}.items():
    print(f" Loading base {base_rev} …")
    m_base = load_model(REPO_BASE, revision=base_rev)
    scores_base = get_induction_scores_robust(m_base)
    release(m_base)

    print(f" Loading PPO model {ppo_repo} …")
    m_ppo = load_model(ppo_repo, revision="main")
    scores_ppo = get_induction_scores_robust(m_ppo)
    release(m_ppo)

    # Signed delta: positive = PPO raised induction score
    delta = scores_ppo - scores_base # [n_layers, n_heads]
    torch.save(delta, f"results_v4/data/analysis_2_delta_{label}vsPPO.pt")
    torch.save(scores_base, f"results_v4/data/analysis_2_base_{label}.pt")
    torch.save(scores_ppo, f"results_v4/data/analysis_2_ppo_{label}.pt")

    n_layers, n_heads = delta.shape
    for layer in range(n_layers):
        for head in range(n_heads):
            ppo_records.append({
                "base": label,
                "layer": layer,
                "head": head,
                "score_base": scores_base[layer, head].item(),
                "score_ppo": scores_ppo[layer, head].item(),
                "delta_induction": delta[layer, head].item(),
                })

            print(f" Δmax={delta.abs().max():.4f} "
                f"Δpos={delta[delta>0].sum():.4f} "
                f"Δneg={delta[delta<0].sum():.4f}")
            print(f" VRAM after release: {vram_gb():.2f} GB")

df_ppo = pd.DataFrame(ppo_records)
df_ppo.to_csv("results_v4/data/analysis_2_ppo_delta.csv", index=False)
print(" Analysis 2 complete - data saved.")

# Visualise signed per-head heatmaps
fig2 = make_subplots(rows=1, cols=2, subplot_titles=("50M+PPO", "200M+PPO"))
for i, base_label in enumerate(["50M", "200M"], 1):
    sub = df_ppo[df_ppo["base"] == base_label]
    pivot = sub.pivot(index="layer", columns="head", values="delta_induction")
    fig2.add_trace(go.Heatmap(z=pivot.values, x=pivot.columns, y=pivot.index, colorscale="RdBu", zmid=0, showscale=(i==2)), row=1, col=i)

fig2.update_yaxes(autorange="reversed")
fig2.update_xaxes(side="top")
fig2.update_layout(title_text="Analysis 2: Signed Δ Induction Score (PPO − Base) | Red=Reinforced | Blue=Suppressed", template="plotly_white")
fig2.write_html("results_v4/figures/analysis_2_ppo_delta.html")
fig2.write_image("results_v4/figures/analysis_2_ppo_delta.png", width=1200, height=500)
fig2.show()

## Analysis 3 - Entity Coreference Tracking

In [ ]:
COREFERENCE_PROBES_V2 = [
    {
        "prompt":  "The boy gave the ball to the girl . The ball belonged to the",
        "correct": " boy",
        "foil":    " girl",
    },
    {
        "prompt":  "The king gave the ring to the queen . The ring belonged to the",
        "correct": " king",
        "foil":    " queen",
    },
    {
        "prompt":  "The dog followed the cat into the wood . The cat hid from the",
        "correct": " dog",
        "foil":    " cat",
    },
    {
        "prompt":  "The man and the woman walked down the road . The woman waved at the",
        "correct": " man",
        "foil":    " woman",
    },
    {
        "prompt":  "John went to the store . John bought",
        "correct": " milk",
        "foil":    " shoes",
    },
    {
        "prompt":  "Mary saw the dog . Mary",
        "correct": " pet",
        "foil":    " hid",
    },
    {
        "prompt":  "Tom read the book . Tom liked",
        "correct": " it",
        "foil":    " nothing",
    },
    {
        "prompt":  "Anna cooked dinner . Anna served",
        "correct": " it",
        "foil":    " them",
    },
    {
        "prompt":  "Sam drove fast . Sam reached",
        "correct": " home",
        "foil":    " work",
    },
]


def preflight_probe_validation(model: HookedTransformer, probes: list[dict]) -> list[dict]:
    valid = []
    for p in probes:
        try:
            validate_single_token(model, p["correct"])
            validate_single_token(model, p["foil"])
            valid.append(p)
        except ValueError as e:
            print(f"    Preflight skip: {e}")
    print(f"  {len(valid)}/{len(probes)} probes passed preflight validation")
    return valid


print("Analysis 3 (v4): Entity Coreference Tracking - Verified Single-Token Probes")
coref_rows = []

for rev in MECH_CHECKPOINTS:
    print(f"  {rev} ...")
    m = load_model(REPO_BASE, revision=rev)

    valid_probes = preflight_probe_validation(m, COREFERENCE_PROBES_V2)

    for probe in valid_probes:
        correct_id = validate_single_token(m, probe["correct"])
        foil_id    = validate_single_token(m, probe["foil"])
        tokens     = m.to_tokens(probe["prompt"])
        with torch.no_grad():
            logits = m(tokens)
        logit_diff = (logits[0, -1, correct_id] - logits[0, -1, foil_id]).item()

        coref_rows.append({
            "checkpoint":     rev.replace("chck_", ""),
            "probe_correct":  probe["correct"].strip(),
            "probe_foil":     probe["foil"].strip(),
            "prompt_snippet": probe["prompt"][-40:],
            "logit_diff":     logit_diff,
            "above_chance":   logit_diff > 0,
        })

    release(m)
    print(f"         VRAM after release: {vram_gb():.2f} GB")

df_coref = pd.DataFrame(coref_rows)
df_coref.to_csv("results_v4/data/analysis_3_coreference_logits.csv", index=False)
print("Analysis 3 complete - data saved.")

coref_summary = df_coref.groupby("checkpoint")["logit_diff"].agg(["mean", "std", "count"])
print("\nCoreference logit diff (mean +/- std) per checkpoint:")
print(coref_summary.to_string())

fig3 = make_subplots(rows=1, cols=2, subplot_titles=("Individual Probes", "Mean Logit Diff (+/-1 SD)"))

for probe in df_coref["probe_correct"].unique():
    sub = df_coref[df_coref["probe_correct"] == probe]
    fig3.add_trace(go.Bar(x=sub["checkpoint"], y=sub["logit_diff"], name=probe), row=1, col=1)

c_sum_reset = coref_summary.reset_index()
fig3.add_trace(go.Scatter(x=c_sum_reset["checkpoint"], y=c_sum_reset["mean"], error_y=dict(type='data', array=c_sum_reset["std"]), mode='lines+markers', name="Mean +/- SD"), row=1, col=2)

fig3.add_hline(y=0, line_dash="dash", line_color="grey", row=1, col=1)
fig3.add_hline(y=0, line_dash="dash", line_color="grey", row=1, col=2)

fig3.update_layout(barmode='group', title_text="Analysis 3 (v4): Coreference Logits (High variance, no systematic progression)", template="plotly_white")
fig3.write_html("results_v4/figures/analysis_3_coreference.html")
fig3.write_image("results_v4/figures/analysis_3_coreference.png", width=1200, height=500)
fig3.show()


def compute_logit_diff(model, probe):
    correct_id = validate_single_token(model, probe["correct"])
    foil_id    = validate_single_token(model, probe["foil"])
    tokens     = model.to_tokens(probe["prompt"])
    with torch.no_grad():
        logits = model(tokens)
        return (logits[0, -1, correct_id] - logits[0, -1, foil_id]).item()


def bootstrap_ci_across_probes(model, probes, n_boot=1000, ci=95):
    per_probe  = np.array([compute_logit_diff(model, p) for p in probes])
    boot_means = np.array([
        np.mean(np.random.choice(per_probe, size=len(per_probe), replace=True))
        for _ in range(n_boot)
    ])
    alpha = (100 - ci) / 2
    return {
        "mean":     np.mean(per_probe),
        "std":      np.std(per_probe, ddof=1),
        "ci_lo":    np.percentile(boot_means, alpha),
        "ci_hi":    np.percentile(boot_means, 100 - alpha),
        "n_probes": len(per_probe),
    }


print("\nComputing bootstrap confidence intervals for coreference logit diffs ...")
bootstrap_rows = []

for rev in MECH_CHECKPOINTS:
    print(f"  Bootstrap for {rev} ...")
    m            = load_model(REPO_BASE, revision=rev)
    valid_probes = preflight_probe_validation(m, COREFERENCE_PROBES_V2)
    ci_result    = bootstrap_ci_across_probes(m, valid_probes, n_boot=1000)
    bootstrap_rows.append({
        "checkpoint": rev.replace("chck_", ""),
        **ci_result,
    })
    release(m)
    print(f"  {rev}: mean={ci_result['mean']:.3f}  95% CI=[{ci_result['ci_lo']:.3f}, {ci_result['ci_hi']:.3f}]")

df_coref_bootstrap = pd.DataFrame(bootstrap_rows)
df_coref_bootstrap.to_csv("results_v4/data/analysis_3_coreference_bootstrap_ci.csv", index=False)
print("Bootstrap CIs saved to analysis_3_coreference_bootstrap_ci.csv")
print(df_coref_bootstrap.to_string(index=False))

## Analysis 4 - Logit Lens KL Depth

In [ ]:
EPS_THRESHOLD = 0.5
TUNED_LENS_CALIB_N = 50


def logit_lens_kl(model: HookedTransformer, prompt: str) -> list[float]:
    tokens = model.to_tokens(prompt)
    _, cache = model.run_with_cache(tokens, names_filter=lambda n: "resid_post" in n)
    with torch.no_grad():
        final_logits = model(tokens)[0, -1]
        final_probs  = F.softmax(final_logits, dim=-1)

    kl_per_layer = []
    for layer in range(model.cfg.n_layers):
        resid    = cache["resid_post", layer][0, -1]
        ln_out   = model.ln_final(resid.unsqueeze(0)).squeeze(0)
        logits_l = ln_out @ model.W_U
        probs_l  = F.softmax(logits_l, dim=-1)
        kl       = F.kl_div(probs_l.log(), final_probs, reduction="sum").item()
        kl_per_layer.append(kl)

    return kl_per_layer


def try_tuned_lens_kl(model: HookedTransformer, prompt: str, calib_n: int = TUNED_LENS_CALIB_N):
    try:
        from tuned_lens.nn import TunedLens
        import random

        calib_tokens = torch.randint(100, EXPECTED_VOCAB, (calib_n, 32), device=DEVICE)
        tl  = TunedLens.from_model(model.hf_model).to(DEVICE)
        opt = torch.optim.Adam(tl.parameters(), lr=1e-3)

        for step in range(200):
            idx  = random.randint(0, calib_n - 1)
            toks = calib_tokens[idx:idx+1]
            _, cache_calib = model.run_with_cache(toks, names_filter=lambda n: "resid_post" in n)
            with torch.no_grad():
                target = F.softmax(model(toks)[0, -1], dim=-1)
            loss = 0.0
            for L in range(model.cfg.n_layers):
                resid = cache_calib["resid_post", L][0, -1]
                pred  = F.softmax(tl(resid, L), dim=-1)
                loss += F.kl_div(pred.log(), target, reduction="sum")
            opt.zero_grad()
            loss.backward()
            opt.step()

        tokens = model.to_tokens(prompt)
        _, cache_probe = model.run_with_cache(tokens, names_filter=lambda n: "resid_post" in n)
        with torch.no_grad():
            final_probs = F.softmax(model(tokens)[0, -1], dim=-1)

        kl_tuned = []
        with torch.no_grad():
            for L in range(model.cfg.n_layers):
                resid = cache_probe["resid_post", L][0, -1]
                probs = F.softmax(tl(resid, L), dim=-1)
                kl    = F.kl_div(probs.log(), final_probs, reduction="sum").item()
                kl_tuned.append(kl)

        del tl
        gc.collect()
        return kl_tuned

    except Exception as e:
        print(f"  Tuned lens skipped ({e.__class__.__name__}: {e})")
        return None


print("Analysis 4: Logit Lens KL Depth + Tuned Lens Cross-Validation")
kl_rows = []

for rev in MECH_CHECKPOINTS:
    print(f"  {rev} ...")
    m = load_model(REPO_BASE, revision=rev)

    klds_std   = logit_lens_kl(m, STORY_PROMPT)
    klds_tuned = try_tuned_lens_kl(m, STORY_PROMPT)

    commit_std   = next((i for i, v in enumerate(klds_std)   if v < EPS_THRESHOLD), m.cfg.n_layers)
    commit_tuned = next((i for i, v in enumerate(klds_tuned) if v < EPS_THRESHOLD), m.cfg.n_layers) if klds_tuned else None

    release(m)
    print(f"  Standard commit_layer={commit_std} | Tuned commit_layer={commit_tuned}")
    print(f"  VRAM after release: {vram_gb():.2f} GB")

    for layer in range(len(klds_std)):
        kl_rows.append({
            "checkpoint":      rev.replace("chck_", ""),
            "layer":           layer,
            "kl_divergence":   klds_std[layer],
            "kl_tuned":        klds_tuned[layer] if klds_tuned else None,
            "commit_standard": commit_std,
            "commit_tuned":    commit_tuned,
        })

df_kl = pd.DataFrame(kl_rows)
df_kl.to_csv("results_v4/data/analysis_4_logit_lens_kl.csv", index=False)
print("Analysis 4 complete - data saved.")

fig4 = make_subplots(rows=1, cols=2, subplot_titles=("Standard Logit Lens", "Tuned Lens"))

for ckpt in df_kl["checkpoint"].unique():
    sub = df_kl[df_kl["checkpoint"] == ckpt]
    fig4.add_trace(go.Scatter(x=sub["layer"], y=sub["kl_divergence"], mode='lines+markers', name=ckpt, legendgroup=ckpt), row=1, col=1)
    if sub["kl_tuned"].notna().any():
        sub_tuned = sub.dropna(subset=["kl_tuned"])
        fig4.add_trace(go.Scatter(x=sub_tuned["layer"], y=sub_tuned["kl_tuned"], mode='lines+markers', name=ckpt, legendgroup=ckpt, showlegend=False), row=1, col=2)

fig4.add_hline(y=EPS_THRESHOLD, line_dash="dot", line_color="red", row=1, col=1)
fig4.add_hline(y=EPS_THRESHOLD, line_dash="dot", line_color="red", row=1, col=2)

fig4.update_layout(title_text="Analysis 4: Logit Lens KL-Divergence per Layer", template="plotly_white")
fig4.write_html("results_v4/figures/analysis_4_logit_lens.html")
fig4.write_image("results_v4/figures/analysis_4_logit_lens.png", width=1200, height=450)
fig4.show()

## Analysis 5 - Attention Entropy Sweep

In [ ]:
def compute_attention_entropy(model: HookedTransformer, prompt: str) -> torch.Tensor:
    tokens = model.to_tokens(prompt)
    _, cache = model.run_with_cache(tokens, names_filter=lambda n: "pattern" in n)

    n_layers = model.cfg.n_layers
    n_heads = model.cfg.n_heads
    ent_matrix = torch.zeros(n_layers, n_heads)

    for layer in range(n_layers):
        pattern = cache["pattern", layer][0] # [n_heads, q_pos, k_pos]
        last_row = pattern[:, -1, :].cpu().numpy() # [n_heads, k_pos]
        for head in range(n_heads):
            p = last_row[head]
            p = np.maximum(p, 1e-10) # numerical stability
            ent_matrix[layer, head] = float(scipy_entropy(p))

            model.reset_hooks()
            return ent_matrix

            print(" Analysis 5: Attention Entropy Sweep")
            entropy_rows = []

            for rev in MECH_CHECKPOINTS:
                print(f" {rev} …")
                m = load_model(REPO_BASE, revision=rev)
                ent = compute_attention_entropy(m, STORY_PROMPT)
                torch.save(ent, f"results_v4/data/analysis_5_entropy_{rev}.pt")

                for layer in range(m.cfg.n_layers):
                    for head in range(m.cfg.n_heads):
                        entropy_rows.append({
                            "checkpoint": rev.replace("chck_", ""),
                            "layer": layer,
                            "head": head,
                            "entropy": ent[layer, head].item(),
                            })

                        release(m)
                        print(f" VRAM after release: {vram_gb():.2f} GB")

                        df_entropy = pd.DataFrame(entropy_rows)
                        df_entropy.to_csv("results_v4/data/analysis_5_entropy.csv", index=False)
                        print(" Analysis 5 complete - data saved.")

                        # Visualise - combined
                        ckpts = list(df_entropy["checkpoint"].unique())
                        fig5 = make_subplots(rows=2, cols=3, subplot_titles=[f"Entropy @ {c}" for c in ckpts] + ["Distribution"])

                        for i, ckpt in enumerate(ckpts):
                            row = (i // 3) + 1
                            col = (i % 3) + 1
                            sub = df_entropy[df_entropy["checkpoint"] == ckpt]
                            pivot = sub.pivot(index="layer", columns="head", values="entropy")
                            fig5.add_trace(go.Heatmap(z=pivot.values, x=pivot.columns, y=pivot.index, colorscale="Viridis_r", showscale=False), row=row, col=col)
                            fig5.update_yaxes(autorange="reversed", row=row, col=col)

                            for ckpt in ckpts:
                                sub = df_entropy[df_entropy["checkpoint"] == ckpt]
                                fig5.add_trace(go.Box(y=sub["entropy"], name=ckpt, showlegend=False), row=2, col=3)

                                fig5.update_layout(title_text="Analysis 5: Attention Entropy (Bifurcation Emerging)", template="plotly_white", height=800)
                                fig5.write_html("results_v4/figures/analysis_5_entropy_combined.html")
                                fig5.write_image("results_v4/figures/analysis_5_entropy_combined.png", width=1200, height=800)
                                fig5.show()

## Analysis 6 - Story Output Diversity

In [ ]:
import itertools
from collections import Counter
import sacrebleu

def compute_output_diversity(model: HookedTransformer, prompt: str, n_samples: int = 20):
    # Prepare generation parameters
    hf_model = model.hf_model
    tokenizer = model.tokenizer
    _orig_padding_side = tokenizer.padding_side # save to restore after
    tokenizer.padding_side = "left"

    inputs = tokenizer(prompt, return_tensors="pt").to(DEVICE)
    prompt_len = inputs.input_ids.shape[1]

    with torch.no_grad():
        outputs = hf_model.generate(
            **inputs,
            max_new_tokens=20,
            do_sample=True,
            temperature=1.0,
            num_return_sequences=n_samples,
            pad_token_id=tokenizer.eos_token_id,
            )

        first_tokens = []
        generations = []

        for i in range(n_samples):
            # Extract the first generated token
            first_tok = outputs[i, prompt_len].item()
            first_tokens.append(first_tok)

            # Decode only the generated completion
            gen_text = tokenizer.decode(outputs[i, prompt_len:], skip_special_tokens=True)
            generations.append(gen_text)

            # 1. Entropy of first generated token distribution
            token_counts = list(Counter(first_tokens).values())
            probs = np.array(token_counts) / n_samples
            first_tok_entropy = float(scipy_entropy(probs))

            # 2. Pairwise Lexical Diversity (mean pairwise BLEU)
            pairwise_bleus = []
            for i, j in itertools.combinations(range(n_samples), 2):
                # Calculate BLEU using smoothing for short sentences
                score = sacrebleu.sentence_bleu(generations[i], [generations[j]], tokenize='13a').score
                pairwise_bleus.append(score)

                mean_pairwise_bleu = np.mean(pairwise_bleus) if pairwise_bleus else 0.0

                tokenizer.padding_side = _orig_padding_side # restore shared tokenizer state
                return first_tok_entropy, mean_pairwise_bleu, generations

                print(" Analysis 6: Story Output Diversity")
                diversity_rows = []

                for rev in MECH_CHECKPOINTS:
                    print(f" {rev} …")
                    m = load_model(REPO_BASE, revision=rev)

                    ent, bleu, _ = compute_output_diversity(m, STORY_PROMPT, n_samples=20)

                    diversity_rows.append({
                        "checkpoint": rev.replace("chck_", ""),
                        "first_tok_entropy": ent,
                        "mean_pairwise_bleu": bleu,
                        })

                    release(m)
                    print(f" First tok entropy: {ent:.3f} | Mean pairwise BLEU: {bleu:.2f}")
                    print(f" VRAM after release: {vram_gb():.2f} GB")

                    df_diversity = pd.DataFrame(diversity_rows)
                    df_diversity.to_csv("results_v4/data/analysis_6_story_diversity.csv", index=False)
                    print(" Analysis 6 complete - data saved.")

                    # Visualise
                    from plotly.subplots import make_subplots
                    fig6 = make_subplots(specs=[[{"secondary_y": True}]])

                    fig6.add_trace(
                        go.Scatter(x=df_diversity["checkpoint"], y=df_diversity["first_tok_entropy"],
                        name="1st-Token Entropy", mode="lines+markers", line=dict(color="blue")),
                        secondary_y=False,
                        )

                    fig6.add_trace(
                        go.Scatter(x=df_diversity["checkpoint"], y=df_diversity["mean_pairwise_bleu"],
                        name="Pairwise BLEU", mode="lines+markers", line=dict(color="red")),
                        secondary_y=True,
                        )

                    fig6.update_layout(
                        title_text="Analysis 6: Story Output Diversity (Mode Collapse vs Variation)",
                        template="plotly_white"
                        )
                    fig6.update_xaxes(title_text="Pretraining Size (words)")
                    fig6.update_yaxes(title_text="Entropy (bits)", secondary_y=False)
                    fig6.update_yaxes(title_text="Mean Pairwise BLEU", secondary_y=True)

                    fig6.write_html("results_v4/figures/analysis_6_story_diversity.html")
                    fig6.write_image("results_v4/figures/analysis_6_story_diversity.png", width=1000, height=500)
                    fig6.show()

## Analysis 7 - Behavioral PPO Impact Across Checkpoints

In [ ]:
def score_coherence(base_model: HookedTransformer, text: str) -> float:
    tokens = base_model.to_tokens(text, prepend_bos=True)
    if tokens.shape[1] <= 1:
        return 0.0
        with torch.no_grad():
            loss = base_model.hf_model(tokens, labels=tokens).loss.item()
            return float(loss)

print(" Analysis 7 (NEW in v4): Behavioral PPO Impact")
v4_rows = []

for label, (base_rev, ppo_repo) in {
    "50M": ("chck_50M", PPO_REPOS["50M+PPO"]),
    "200M": ("chck_200M", PPO_REPOS["200M+PPO"]),
}.items():
    print(f" {label} evaluation …")

    # 1. Base model stats
    m_base = load_model(REPO_BASE, revision=base_rev)
    ent_base, bleu_base, gen_base = compute_output_diversity(m_base, STORY_PROMPT, n_samples=50)
    coh_base = np.mean([score_coherence(m_base, g) for g in gen_base])
    len_base = np.mean([len(m_base.tokenizer.encode(g)) for g in gen_base])

    # 2. PPO model stats
    m_ppo = load_model(ppo_repo, revision="main")
    ent_ppo, bleu_ppo, gen_ppo = compute_output_diversity(m_ppo, STORY_PROMPT, n_samples=50)
    coh_ppo = np.mean([score_coherence(m_base, g) for g in gen_ppo])
    len_ppo = np.mean([len(m_ppo.tokenizer.encode(g)) for g in gen_ppo])

    release(m_ppo)
    release(m_base)

    v4_rows.append({
        "checkpoint": label,
        "stage": "base",
        "first_tok_entropy": ent_base,
        "mean_pairwise_bleu": bleu_base,
        "mean_story_length": len_base,
        "coherence_loss": coh_base,
        })

    v4_rows.append({
        "checkpoint": label,
        "stage": "ppo",
        "first_tok_entropy": ent_ppo,
        "mean_pairwise_bleu": bleu_ppo,
        "mean_story_length": len_ppo,
        "coherence_loss": coh_ppo,
        })

df_v4 = pd.DataFrame(v4_rows)
df_v4.to_csv("results_v4/data/analysis_7_v4_diversity_comparison.csv", index=False)

# Compute deltas
delta_rows = []
for label in ["50M", "200M"]:
    sub = df_v4[df_v4["checkpoint"] == label].set_index("stage")
    delta_rows.append({
        "checkpoint": label,
        "delta_first_tok_entropy": sub.loc["ppo", "first_tok_entropy"] - sub.loc["base", "first_tok_entropy"],
        "delta_pairwise_bleu": sub.loc["ppo", "mean_pairwise_bleu"] - sub.loc["base", "mean_pairwise_bleu"],
        "delta_story_length": sub.loc["ppo", "mean_story_length"] - sub.loc["base", "mean_story_length"],
        "delta_coherence": sub.loc["ppo", "coherence_loss"] - sub.loc["base", "coherence_loss"],
        })

df_delta = pd.DataFrame(delta_rows)
df_delta.to_csv("results_v4/data/analysis_7_v4_diversity_delta.csv", index=False)
print(" Analysis 7 complete - data saved.")

# Visualise A7 (Before/After)
fig7 = make_subplots(rows=2, cols=2, subplot_titles=(
    "1st-Token Entropy (Base vs PPO)", "Pairwise BLEU (Base vs PPO)",
    "Story Length (Base vs PPO)", "Coherence Loss (Base vs PPO)"
))

for label in ["50M", "200M"]:
    sub = df_v4[df_v4["checkpoint"] == label]
    fig7.add_trace(go.Bar(name=f"{label} Base", x=sub["stage"], y=sub["first_tok_entropy"], marker_color=["#1f77b4" if s == "base" else "#ff7f0e" for s in sub["stage"]], legendgroup=label, showlegend=False), row=1, col=1)
    fig7.add_trace(go.Bar(name=f"{label} PPO", x=sub["stage"], y=sub["mean_pairwise_bleu"], marker_color=["#1f77b4" if s == "base" else "#ff7f0e" for s in sub["stage"]], legendgroup=label, showlegend=False), row=1, col=2)
    fig7.add_trace(go.Bar(name=f"{label} Base", x=sub["stage"], y=sub["mean_story_length"], marker_color=["#1f77b4" if s == "base" else "#ff7f0e" for s in sub["stage"]], legendgroup=label, showlegend=False), row=2, col=1)
    fig7.add_trace(go.Bar(name=f"{label} PPO", x=sub["stage"], y=sub["coherence_loss"], marker_color=["#1f77b4" if s == "base" else "#ff7f0e" for s in sub["stage"]], legendgroup=label, showlegend=False), row=2, col=2)

fig7.update_layout(title_text="Analysis 7: Behavioral Impact of PPO at 50M vs 200M", template="plotly_white", barmode='group')
fig7.write_html("results_v4/figures/analysis_7_v4_behavioral_impact.html")
fig7.write_image("results_v4/figures/analysis_7_v4_behavioral_impact.png", width=1000, height=800)
fig7.show()

## Analysis 8 - Direct Logit Attribution on Coreference Probes

In [ ]:
print(" Analysis 8: Direct Logit Attribution (DLA) on Coreference Probes")

# Coreference probes used for DLA (subset with clear entity tracking)
DLA_PROBES = [
    {"prompt": "The dog followed the cat into the wood . The cat hid from the",
    "correct": " dog", "foil": " cat"},
    {"prompt": "The boy gave the ball to the girl . The ball belonged to the",
    "correct": " boy", "foil": " girl"},
    {"prompt": "The king gave the ring to the queen . The ring belonged to the",
    "correct": " king", "foil": " queen"},
    {"prompt": "The man and the woman walked down the road . The woman waved at the",
    "correct": " man", "foil": " woman"},
    ]

DLA_CHECKPOINTS = ["chck_10M", "chck_50M", "chck_200M", "chck_1000M"]

dla_per_probe_rows = []
dla_avg_rows = []

for rev in DLA_CHECKPOINTS:
    print(f" DLA for {rev} …")
    m = load_model(REPO_BASE, revision=rev)

    n_layers = m.cfg.n_layers
    n_heads = m.cfg.n_heads
    d_head = m.cfg.d_head
    d_model = m.cfg.d_model

    # Accumulate per-head DLA across probes
    probe_dlas = []

    for probe in DLA_PROBES:
        correct_id = validate_single_token(m, probe["correct"])
        foil_id = validate_single_token(m, probe["foil"])
        tokens = m.to_tokens(probe["prompt"])

        # Forward pass with cache
        logits, cache = m.run_with_cache(tokens)

        # Direction vector: correct - foil in unembedding space
        direction = m.W_U[:, correct_id] - m.W_U[:, foil_id] # (d_model,)

        # ln_final correction: get scale from the cached ln_final output
        resid_pre_ln = cache["resid_post", n_layers - 1][0, -1] # (d_model,)
        ln_scale = m.ln_final(resid_pre_ln.unsqueeze(0)).squeeze(0) / (resid_pre_ln + 1e-8)

        per_head_dla = np.zeros((n_layers, n_heads))

        for layer in range(n_layers):
            z = cache[f"blocks.{layer}.attn.hook_z"][0, -1] # (n_heads, d_head)
            W_O = m.W_O[layer] # (n_heads, d_head, d_model)

            for head in range(n_heads):
                # Project from d_head to d_model via W_O
                head_output = z[head] @ W_O[head] # (d_model,)
                # Apply ln_final correction
                head_output_corrected = head_output * ln_scale
                # DLA = dot product with direction
                dla_val = (head_output_corrected @ direction).item()
                per_head_dla[layer, head] = dla_val

                dla_per_probe_rows.append({
                    "checkpoint": rev.replace("chck_", ""),
                    "probe": f"{probe['correct'].strip()}/{probe['foil'].strip()}",
                    "layer": layer,
                    "head": head,
                    "dla": dla_val,
                    })

                probe_dlas.append(per_head_dla)

                # Average across probes
                avg_dla = np.mean(probe_dlas, axis=0)
                for layer in range(n_layers):
                    for head in range(n_heads):
                        dla_avg_rows.append({
                            "checkpoint": rev.replace("chck_", ""),
                            "layer": layer,
                            "head": head,
                            "dla_mean": avg_dla[layer, head],
                            })

                        release(m)
                        print(f" VRAM after release: {vram_gb():.2f} GB")

                        df_dla_per_probe = pd.DataFrame(dla_per_probe_rows)
                        df_dla_avg = pd.DataFrame(dla_avg_rows)
                        df_dla_per_probe.to_csv("results_v4/data/analysis_8_dla_per_probe.csv", index=False)
                        df_dla_avg.to_csv("results_v4/data/analysis_8_dla_averaged.csv", index=False)
                        print(" Analysis 8 (DLA) complete - data saved.")

                        # Visualise DLA heatmaps
                        from plotly.subplots import make_subplots

                        fig_dla = make_subplots(
                            rows=1, cols=len(DLA_CHECKPOINTS),
                            subplot_titles=[f"{c.replace('chck_','')}" for c in DLA_CHECKPOINTS],
                            shared_yaxes=True,
                            )

                        for i, rev in enumerate(DLA_CHECKPOINTS):
                            sub = df_dla_avg[df_dla_avg["checkpoint"] == rev.replace("chck_", "")]
                            pivot = sub.pivot(index="layer", columns="head", values="dla_mean")
                            fig_dla.add_trace(
                                go.Heatmap(z=pivot.values, x=[f"H{h}" for h in pivot.columns],
                                y=[f"L{l}" for l in pivot.index],
                                colorscale="RdBu_r", zmid=0, showscale=(i == len(DLA_CHECKPOINTS) - 1)),
                                row=1, col=i + 1,
                                )

                            fig_dla.update_layout(title_text="Analysis 8: Direct Logit Attribution (Coreference, 4-probe avg)",
                                template="plotly_white", height=500, width=1200)
                            fig_dla.write_html("results_v4/figures/analysis_8_dla_heatmap.html")
                            fig_dla.write_image("results_v4/figures/analysis_8_dla_heatmap.png", width=1200, height=500)
                            fig_dla.show()


## Analysis 9 - PPO Reward vs L8H0 Induction Correlation

In [ ]:
print(" Analysis 9: PPO Reward vs L8H0 Induction Correlation")
from scipy.stats import spearmanr, pearsonr

# Step 1: Generate stories from 200M+PPO
print(" Step 1: Generating 50 stories from 200M+PPO …")
m_ppo = load_model(PPO_REPOS["200M+PPO"], revision="main")
tokenizer = m_ppo.tokenizer
tokenizer.pad_token = tokenizer.eos_token

input_ids = tokenizer(STORY_PROMPT, return_tensors="pt")["input_ids"].to(DEVICE)

stories = []
for i in range(50):
    with torch.no_grad():
        out = m_ppo.hf_model.generate(
            input_ids,
            max_new_tokens=100,
            do_sample=True,
            temperature=0.9,
            top_k=50,
            pad_token_id=tokenizer.eos_token_id,
            )
        story_text = tokenizer.decode(out[0], skip_special_tokens=True)
        stories.append(story_text)
        if (i + 1) % 10 == 0:
            print(f" Generated {i + 1}/50")

            # Step 2: Measure L8H0 induction attention
            print(" Step 2: Measuring L8H0 induction attention …")
            l8h0_scores = []

            for story_text in stories:
                tokens = m_ppo.to_tokens(story_text)
                seq_len = tokens.shape[1]

                with torch.no_grad():
                    _, cache = m_ppo.run_with_cache(tokens)

                    # L8H0 attention pattern: (seq_len, seq_len)
                    attn = cache["blocks.8.attn.hook_pattern"][0, 0] # head 0

                    # Induction = fraction of attention from final position to positions
                    # where the token at pos-1 matches any earlier token (repeated-token positions)
                    token_list = tokens[0].cpu().tolist()
                    repeated_positions = set()
                    for pos in range(1, seq_len):
                        prev_token = token_list[pos - 1]
                        for earlier in range(pos):
                            if token_list[earlier] == prev_token and (earlier + 1) < seq_len:
                                repeated_positions.add(earlier + 1)

                                if len(repeated_positions) > 0:
                                    attn_last = attn[-1].cpu().numpy()
                                    induction_frac = sum(attn_last[p] for p in repeated_positions if p < seq_len)
                                else:
                                    induction_frac = 0.0

                                    l8h0_scores.append(float(induction_frac))

                                    release(m_ppo)
                                    print(f" VRAM after release: {vram_gb():.2f} GB")

                                    # Step 3: Score stories with Llama3
                                    print(" Step 3: Scoring stories with Llama-3.1-8B-Instruct …")
                                    from transformers import AutoModelForCausalLM, AutoTokenizer as HFAutoTokenizer

                                    TEACHER_MODEL_ID = "meta-llama/Llama-3.1-8B-Instruct"
                                    teacher_tokenizer = HFAutoTokenizer.from_pretrained(TEACHER_MODEL_ID)
                                    teacher_model = AutoModelForCausalLM.from_pretrained(
                                        TEACHER_MODEL_ID,
                                        torch_dtype=torch.bfloat16,
                                        device_map="auto",
                                        )

                                    TEACHER_PROMPT_TEMPLATE = """You are evaluating a children's story. Score the following story on three criteria, each from 0 to 3:

                                    1. **Readability** (0-3): Is the text grammatically correct and easy to follow?
                                    2. **Narrative** (0-3): Does the story have a coherent narrative structure?
                                    3. **Creativity** (0-3): Is the story creative and engaging?

                                    Story:
                                    {story}

                                    Provide your scores in the format:
                                    Readability: X
                                    Narrative: X
                                    Creativity: X"""

                                    reward_scores = []
                                    for i, story in enumerate(stories):
                                        prompt = TEACHER_PROMPT_TEMPLATE.format(story=story)
                                        messages = [{"role": "user", "content": prompt}]
                                        input_text = teacher_tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
                                        input_ids = teacher_tokenizer(input_text, return_tensors="pt").input_ids.to(teacher_model.device)

                                        with torch.no_grad():
                                            output = teacher_model.generate(input_ids, max_new_tokens=100, do_sample=False)

                                            response = teacher_tokenizer.decode(output[0][input_ids.shape[1]:], skip_special_tokens=True)

                                            # Parse scores
                                            scores_found = re.findall(r"\b\d+\b", response)
                                            if len(scores_found) >= 3:
                                                raw = [min(max(int(s), 0), 3) for s in scores_found[:3]]
                                                reward = np.mean(raw) / 3.0 # normalize to [0, 1]
                                            else:
                                                reward = 0.5 # fallback

                                                reward_scores.append(reward)
                                                if (i + 1) % 10 == 0:
                                                    print(f" Scored {i + 1}/50")

                                                    # Clean up teacher model
                                                    del teacher_model, teacher_tokenizer
                                                    gc.collect()
                                                    if DEVICE == "cuda":
                                                        torch.cuda.empty_cache()

                                                        # Step 4: Correlation
                                                        print(" Step 4: Computing correlations …")
                                                        spearman_r, spearman_p = spearmanr(l8h0_scores, reward_scores)
                                                        pearson_r, pearson_p = pearsonr(l8h0_scores, reward_scores)

                                                        print(f" Spearman r = {spearman_r:.4f}, p = {spearman_p:.4f}")
                                                        print(f" Pearson r = {pearson_r:.4f}, p = {pearson_p:.4f}")

                                                        # Save results
                                                        df_a9 = pd.DataFrame({
                                                            "story_idx": list(range(50)),
                                                            "l8h0_induction_frac": l8h0_scores,
                                                            "teacher_reward": reward_scores,
                                                            })
                                                        df_a9.to_csv("results_v4/data/analysis_9_ppo_reward_vs_l8h0.csv", index=False)

                                                        corr_summary = pd.DataFrame([{
                                                            "spearman_r": spearman_r, "spearman_p": spearman_p,
                                                            "pearson_r": pearson_r, "pearson_p": pearson_p,
                                                            "n_stories": 50,
                                                            }])
                                                        corr_summary.to_csv("results_v4/data/analysis_9_correlation_summary.csv", index=False)
                                                        print(" Analysis 9 complete - data saved.")

                                                        # Visualise
                                                        fig9 = go.Figure()
                                                        fig9.add_trace(go.Scatter(
                                                            x=l8h0_scores, y=reward_scores, mode="markers",
                                                            marker=dict(size=8, opacity=0.7),
                                                            text=[f"Story {i}" for i in range(50)],
                                                            hoverinfo="text+x+y",
                                                            ))
                                                        fig9.update_layout(
                                                            title=f"Analysis 9: L8H0 Induction vs Teacher Reward (Spearman r={spearman_r:.3f}, p={spearman_p:.3f})",
                                                            xaxis_title="L8H0 Induction Fraction",
                                                            yaxis_title="Teacher Reward (normalized)",
                                                            template="plotly_white",
                                                            )
                                                        fig9.write_html("results_v4/figures/analysis_9_l8h0_vs_reward.html")
                                                        fig9.write_image("results_v4/figures/analysis_9_l8h0_vs_reward.png", width=800, height=500)
                                                        fig9.show()


## Analysis 10 - L8H0 Attention Pattern Across Training

In [ ]:
print(" Analysis 10: L8H0 Attention Pattern Visualization")

L8H0_SENTENCE = "The dog followed the cat into the wood . The cat hid from the"
L8H0_CHECKPOINTS = ["chck_10M", "chck_50M", "chck_200M", "chck_1000M"]

attn_rows = []

for rev in L8H0_CHECKPOINTS:
    print(f" {rev} …")
    m = load_model(REPO_BASE, revision=rev)
    tokens = m.to_tokens(L8H0_SENTENCE)
    str_tokens = m.to_str_tokens(tokens[0])

    with torch.no_grad():
        _, cache = m.run_with_cache(tokens)

        # L8H0 attention from final position to all keys
        attn_pattern = cache["blocks.8.attn.hook_pattern"][0, 0] # (seq, seq)
        final_attn = attn_pattern[-1].cpu().numpy() # attention from last position

        for pos, (tok, attn_val) in enumerate(zip(str_tokens, final_attn)):
            attn_rows.append({
                "checkpoint": rev.replace("chck_", ""),
                "position": pos,
                "token": tok,
                "attention": float(attn_val),
                })

            release(m)
            print(f" VRAM after release: {vram_gb():.2f} GB")

            df_l8h0_attn = pd.DataFrame(attn_rows)
            df_l8h0_attn.to_csv("results_v4/data/analysis_10_l8h0_attention_pattern.csv", index=False)
            print(" Analysis 10 complete - data saved.")

            # Visualise
            fig10 = make_subplots(
                rows=1, cols=len(L8H0_CHECKPOINTS),
                subplot_titles=[c.replace("chck_", "") for c in L8H0_CHECKPOINTS],
                shared_yaxes=True,
                )

            for i, rev in enumerate(L8H0_CHECKPOINTS):
                sub = df_l8h0_attn[df_l8h0_attn["checkpoint"] == rev.replace("chck_", "")]
                fig10.add_trace(
                    go.Bar(x=sub["token"], y=sub["attention"], name=rev.replace("chck_", ""),
                    showlegend=False),
                    row=1, col=i + 1,
                    )

                fig10.update_layout(
                    title_text="Analysis 10: L8H0 Attention from Final Position (Coreference Sentence)",
                    template="plotly_white", height=400, width=1400,
                    )
                fig10.update_yaxes(title_text="Attention", row=1, col=1)
                fig10.write_html("results_v4/figures/analysis_10_l8h0_attention.html")
                fig10.write_image("results_v4/figures/analysis_10_l8h0_attention.png", width=1400, height=400)
                fig10.show()


In [ ]:
commit_lookup_std = df_kl.groupby("checkpoint")["commit_standard"].first().to_dict()
commit_lookup_tun = df_kl.groupby("checkpoint")["commit_tuned"].first().to_dict()
peak_ind = df_induction.set_index("checkpoint")["max_score"].to_dict()
coref_mean_by_ckpt = df_coref.groupby("checkpoint")["logit_diff"].mean().to_dict()
entropy_median = df_entropy.groupby("checkpoint")["entropy"].median().to_dict()

summary_rows = []
for rev in ["10M", "20M", "50M", "200M", "1000M"]:
    summary_rows.append({
        "Checkpoint": f"{rev} words",
        "Peak Induction Score": f"{peak_ind.get(rev, ' - '):.3f}"
        if isinstance(peak_ind.get(rev), float) else " - ",
        "KL Commit Layer (std)": commit_lookup_std.get(rev, " - "),
        "KL Commit Layer (tuned)": commit_lookup_tun.get(rev, " - "),
        "Coref Logit Diff (mean)": f"{coref_mean_by_ckpt.get(rev, ' - '):.3f}"
        if isinstance(coref_mean_by_ckpt.get(rev), float) else " - ",
        "Attn Entropy (median)": f"{entropy_median.get(rev, ' - '):.3f}"
        if isinstance(entropy_median.get(rev), float) else " - ",
        "1st-Tok Entropy": f"{df_diversity.set_index('checkpoint')['first_tok_entropy'].to_dict().get(rev, ' - '):.3f}"
        if 'df_diversity' in locals() and rev in df_diversity['checkpoint'].values else " - ",
        "Pairwise BLEU": f"{df_diversity.set_index('checkpoint')['mean_pairwise_bleu'].to_dict().get(rev, ' - '):.2f}"
        if 'df_diversity' in locals() and rev in df_diversity['checkpoint'].values else " - ",
        })

    for base_label in ["50M", "200M"]:
        sub = df_ppo[df_ppo["base"] == base_label]
        top_delta = sub["delta_induction"].abs().max()
        n_pos = (sub["delta_induction"] > 0).sum()
        n_neg = (sub["delta_induction"] < 0).sum()
        summary_rows.append({
            "Checkpoint": f"{base_label}+PPO (delta)",
            "Peak Induction Score": f"Δmax={top_delta:.3f} {n_pos} {n_neg} heads",
            "KL Commit Layer (std)": "n/a",
            "KL Commit Layer (tuned)": "n/a",
            "Coref Logit Diff (mean)": "n/a",
            "Attn Entropy (median)": "n/a",
            "1st-Tok Entropy": "n/a",
            "Pairwise BLEU": "n/a",
            })

        df_summary = pd.DataFrame(summary_rows)
        df_summary.to_csv("results_v4/data/summary_table_v4.csv", index=False)
        print(df_summary.to_string(index=False))
        print(" Summary table saved.")

In [ ]:
shutil.make_archive("Kind_LM_Project_B_Results_v4", "zip", "results_v4")

try:
    from google.colab import files
    files.download("Kind_LM_Project_B_Results_v4.zip")
except ImportError:
    pass